# Dataset Exploration: CelebA & CIELAB Color Space

This notebook demonstrates:
1. Loading CelebA celebrity face images.
2. Decomposing RGB images into Lightness (L) and Chrominance (a, b) channels.
3. Channel value distributions and properties.
4. Testing PyTorch `CelebADataset` and `DataLoader`.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

# Ensure project root is in sys.path
sys.path.append(os.path.abspath('..'))

from preprocessing.color_space import rgb_to_lab, lab_to_rgb
from data.celebA_dataset import CelebADataset, get_dataloaders
from preprocessing.prepare_dataset import generate_sample_dataset

print("Environment initialized successfully!")

## 1. Inspect or Generate Sample CelebA Images

In [ ]:
sample_dir = '../data/raw'
if not os.path.exists(sample_dir) or len(os.listdir(sample_dir)) == 0:
    generate_sample_dataset(sample_dir, num_samples=20, image_size=128)

sample_files = [os.path.join(sample_dir, f) for f in os.listdir(sample_dir) if f.endswith(('.jpg', '.png'))]
print(f"Found {len(sample_files)} images.")

## 2. CIELAB Decomposition (L, a, b Channels)

In [ ]:
sample_img = Image.open(sample_files[0]).convert('RGB').resize((128, 128))
rgb_np = np.array(sample_img)

L, ab = rgb_to_lab(rgb_np)
a_channel = ab[:, :, 0]
b_channel = ab[:, :, 1]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(rgb_np)
axes[0].set_title("Original RGB")
axes[0].axis("off")

axes[1].imshow(L, cmap='gray')
axes[1].set_title("L Channel (Lightness)")
axes[1].axis("off")

axes[2].imshow(a_channel, cmap='RdYlGn_r')
axes[2].set_title("a Channel (Green - Red)")
axes[2].axis("off")

axes[3].imshow(b_channel, cmap='YlGnBu')
axes[3].set_title("b Channel (Blue - Yellow)")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 3. Verify Reconstruction Fidelity (LAB -> RGB)

In [ ]:
reconstructed_rgb = lab_to_rgb(L, ab)
diff = np.abs(rgb_np.astype(np.float32) - reconstructed_rgb.astype(np.float32))

print(f"Max absolute pixel difference: {diff.max():.2f}")
print(f"Mean absolute pixel difference: {diff.mean():.4f}")
assert diff.mean() < 2.0, "Reconstruction difference should be negligible!"

## 4. PyTorch Dataset & Batch Inspection

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(sample_dir, batch_size=4, image_size=128)
batch = next(iter(train_loader))

print(f"L channel batch shape:  {batch['L'].shape}  (Min: {batch['L'].min():.2f}, Max: {batch['L'].max():.2f})")
print(f"ab channels batch shape: {batch['ab'].shape} (Min: {batch['ab'].min():.2f}, Max: {batch['ab'].max():.2f})")
print(f"RGB batch shape:        {batch['rgb'].shape} (Min: {batch['rgb'].min():.2f}, Max: {batch['rgb'].max():.2f})")